# Notebook 1: Data Generation and Loading

This notebook generates synthetic customers, products, categories and tags using `Faker`, then loads everything into an Aura Free instance as a graph ready for recommendation queries.

## Before running the notebook

Export the following in your shell:

```bash
export NEO4J_URI="neo4j+s://xxxx.databases.neo4j.io"
export NEO4J_USERNAME="your_username_here"
export NEO4J_PASSWORD="your_password_here"
```

## 1. Install dependencies

In [1]:
%pip install faker==40.36.0 \
             neo4j==6.2.0 \
             tabulate==0.10.0 \
             tqdm==4.67.1 --quiet

print("Install complete.")

Note: you may need to restart the kernel to use updated packages.
Install complete.


## 2. Imports

In [2]:
import os
import random
import sys
import uuid

from datetime import datetime, timedelta
from faker import Faker
from neo4j import GraphDatabase
from tabulate import tabulate
from tqdm import tqdm

In [3]:
fake = Faker()
random.seed(42)
Faker.seed(42)

## 3. Credentials

In [4]:
NEO4J_URI      = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")

print("Credentials set.")

Credentials set.


## 4. Neo4j connection

In [5]:
driver = GraphDatabase.driver(
    NEO4J_URI,
    auth = (NEO4J_USERNAME, NEO4J_PASSWORD)
)

print(driver.verify_connectivity())  # None is expected
print("Connection created.")

None
Connection created.


## 5. Clear database

In [6]:
with driver.session() as session:
    session.run("MATCH (n) DETACH DELETE n")

print("Database cleared.")

Database cleared.


## 6. Constraints and indexes

In [7]:
constraints = [
    "CREATE CONSTRAINT IF NOT EXISTS FOR (c:Customer) REQUIRE c.id   IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (p:Product)  REQUIRE p.id   IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (c:Category) REQUIRE c.name IS UNIQUE",
    "CREATE CONSTRAINT IF NOT EXISTS FOR (t:Tag)      REQUIRE t.name IS UNIQUE",
]

with driver.session() as session:
    for cypher in constraints:
        session.run(cypher)

print("Constraints created.")

Constraints created.


## 7. Define categories, tags and product vocabulary

In [8]:
CATEGORIES = [
    "Electronics", "Clothing", "Books", "Home & Garden", "Sports",
    "Toys", "Beauty", "Automotive", "Food & Drink", "Music",
    "Office", "Pet Supplies", "Travel", "Health", "Outdoor"
]

TAGS = [
    "wireless", "waterproof", "bestseller", "eco-friendly", "portable",
    "handmade", "organic", "limited-edition", "fast-charging", "lightweight",
    "durable", "compact", "smart", "vintage", "premium",
    "budget-friendly", "gift-idea", "new-arrival", "sale", "bundle",
    "noise-canceling", "rechargeable", "foldable", "adjustable", "multi-use",
    "biodegradable", "travel-size", "professional", "kids-safe", "unisex"
]

# Per-category vocabulary for realistic product names and descriptions
PRODUCT_VOCAB = {
    "Electronics": {
        "adjectives": ["Wireless", "Smart", "Portable", "Rechargeable", "Ultra-Slim"],
        "nouns":      ["Headphones", "Bluetooth Speaker", "USB-C Charger", "Mechanical Keyboard", "Webcam", "LED Desk Lamp", "Smartwatch", "Noise-Canceling Earbuds"],
        "use_cases":  ["everyday use", "home office setups", "commuters", "remote workers", "gamers"],
        "benefits":   ["Delivers crisp sound and long battery life", "Pairs seamlessly with all major devices", "Features fast charging and durable build quality", "Designed for all-day comfort"]
    },
    "Clothing": {
        "adjectives": ["Slim-Fit", "Lightweight", "Waterproof", "Breathable", "Stretch"],
        "nouns":      ["Running Jacket", "Joggers", "Hoodie", "Polo Shirt", "Hiking Vest", "Chino Trousers", "Fleece Pullover", "Puffer Coat"],
        "use_cases":  ["outdoor activities", "casual everyday wear", "gym sessions", "travel", "layering in cold weather"],
        "benefits":   ["Made from moisture-wicking fabric", "Keeps you warm without the bulk", "Available in a range of versatile colours", "Machine washable and quick-drying"]
    },
    "Books": {
        "adjectives": ["Mastering", "The Complete Guide to", "An Introduction to", "The Art of", "Practical"],
        "nouns":      ["Python Programming", "Data Science", "Personal Finance", "Leadership", "Mindfulness", "Photography", "Machine Learning", "Public Speaking"],
        "use_cases":  ["self-paced learning", "professionals looking to upskill", "beginners and enthusiasts", "students and researchers"],
        "benefits":   ["Covers both theory and hands-on practice", "Written by industry experts with real-world experience", "Packed with exercises, examples, and case studies", "Clear and accessible for all skill levels"]
    },
    "Home & Garden": {
        "adjectives": ["Bamboo", "Stainless Steel", "Foldable", "Non-Stick", "Ceramic"],
        "nouns":      ["Chopping Board", "Plant Pot", "Storage Box", "Knife Set", "Colander", "Salad Bowl", "Herb Garden Kit", "Compost Bin"],
        "use_cases":  ["modern kitchens", "small spaces", "garden enthusiasts", "everyday cooking", "home organisation"],
        "benefits":   ["Made from sustainable and food-safe materials", "Easy to clean and dishwasher safe", "Compact design fits neatly in any cupboard", "Built to last with reinforced construction"]
    },
    "Sports": {
        "adjectives": ["Pro-Grade", "Lightweight", "Adjustable", "High-Performance", "Durable"],
        "nouns":      ["Yoga Mat", "Resistance Bands Set", "Foam Roller", "Dumbbell Pair", "Jump Rope", "Gym Gloves", "Water Bottle", "Cycling Helmet"],
        "use_cases":  ["home workouts", "gym training", "yoga and pilates", "outdoor fitness", "recovery sessions"],
        "benefits":   ["Non-slip surface for safe and stable training", "Suitable for all fitness levels", "Compact and easy to store", "Made from sweat-resistant and easy-clean materials"]
    },
    "Toys": {
        "adjectives": ["Educational", "Interactive", "Wooden", "Battery-Free", "Creative"],
        "nouns":      ["Building Blocks Set", "Puzzle Game", "Science Kit", "Art & Craft Set", "Remote Control Car", "Board Game", "Magnetic Drawing Board", "Coding Robot"],
        "use_cases":  ["children aged 3 and up", "family game nights", "STEM learning at home", "independent creative play"],
        "benefits":   ["Encourages problem-solving and creativity", "Made from non-toxic and child-safe materials", "Suitable for solo and group play", "Helps develop fine motor skills and concentration"]
    },
    "Beauty": {
        "adjectives": ["Hydrating", "Natural", "Vegan", "Cruelty-Free", "Dermatologist-Tested"],
        "nouns":      ["Face Serum", "Moisturising Cream", "Lip Balm", "Eye Cream", "Shampoo", "Body Lotion", "Sunscreen SPF50", "Vitamin C Toner"],
        "use_cases":  ["daily skincare routines", "sensitive skin", "anti-ageing care", "all skin types"],
        "benefits":   ["Formulated with natural botanicals and no harsh chemicals", "Absorbs quickly without leaving a greasy residue", "Clinically tested for safety and effectiveness", "Suitable for use morning and night"]
    },
    "Automotive": {
        "adjectives": ["Heavy-Duty", "Universal", "Compact", "Waterproof", "Professional-Grade"],
        "nouns":      ["Car Vacuum Cleaner", "Dash Cam", "Tyre Inflator", "Seat Covers Set", "Car Phone Mount", "Jump Starter", "Windscreen Sun Shade", "OBD2 Diagnostic Scanner"],
        "use_cases":  ["everyday drivers", "long road trips", "vehicle maintenance", "fleet owners"],
        "benefits":   ["Compatible with most vehicle makes and models", "Easy to install with no tools required", "Built for all-weather performance", "Compact enough to store in the glove box"]
    },
    "Food & Drink": {
        "adjectives": ["Organic", "Cold-Pressed", "Artisan", "Single-Origin", "Small-Batch"],
        "nouns":      ["Ground Coffee", "Green Tea", "Extra Virgin Olive Oil", "Raw Honey", "Protein Granola", "Dark Chocolate", "Hot Sauce", "Herbal Tea Blend"],
        "use_cases":  ["morning routines", "healthy snacking", "gourmet cooking", "gifting"],
        "benefits":   ["Sourced from sustainable and ethical producers", "No artificial additives or preservatives", "Rich in antioxidants and natural nutrients", "Certified organic and responsibly packaged"]
    },
    "Music": {
        "adjectives": ["Professional", "Beginner-Friendly", "Acoustic", "Electric", "Portable"],
        "nouns":      ["Guitar Tuner", "Capo", "Drum Practice Pad", "Keyboard Stand", "Microphone", "Guitar Strap", "Sheet Music Stand", "Metronome"],
        "use_cases":  ["practice sessions", "live performances", "home studios", "beginners learning an instrument"],
        "benefits":   ["Precise and reliable with a fast response time", "Compatible with a wide range of instruments", "Lightweight and easy to transport", "Built to withstand regular stage use"]
    },
    "Office": {
        "adjectives": ["Ergonomic", "Adjustable", "Minimalist", "Heavy-Duty", "Compact"],
        "nouns":      ["Desk Organiser", "Lumbar Support Cushion", "Monitor Stand", "Cable Management Box", "Whiteboard", "Desk Pad", "Stapler", "Letter Tray"],
        "use_cases":  ["home offices", "corporate workspaces", "students", "remote workers"],
        "benefits":   ["Helps reduce desk clutter and improve focus", "Promotes better posture during long working hours", "Easy to assemble and fits most desk sizes", "Made from durable and premium-feel materials"]
    },
    "Pet Supplies": {
        "adjectives": ["Durable", "Washable", "Grain-Free", "Veterinary-Approved", "Eco-Friendly"],
        "nouns":      ["Dog Harness", "Cat Scratching Post", "Pet Water Fountain", "Orthopedic Pet Bed", "Interactive Puzzle Feeder", "Retractable Lead", "Grooming Brush", "Catnip Toy"],
        "use_cases":  ["dogs and cats of all sizes", "indoor pets", "active dogs", "senior pets needing extra comfort"],
        "benefits":   ["Made from non-toxic and pet-safe materials", "Easy to clean and maintain", "Designed to reduce anxiety and promote wellbeing", "Tough enough to withstand enthusiastic chewers"]
    },
    "Travel": {
        "adjectives": ["Lightweight", "Carry-On", "Waterproof", "Expandable", "Anti-Theft"],
        "nouns":      ["Cabin Suitcase", "Packing Cubes Set", "Travel Pillow", "Passport Holder", "Toiletry Bag", "Luggage Scale", "Compression Sacks", "Travel Adapter"],
        "use_cases":  ["frequent flyers", "weekend breaks", "backpacking trips", "business travel"],
        "benefits":   ["TSA-approved and airline carry-on compliant", "Maximises packing space without adding weight", "RFID-blocking lining protects your cards and passport", "Built from tear-resistant and water-repellent fabric"]
    },
    "Health": {
        "adjectives": ["Clinically-Tested", "Natural", "High-Strength", "Vegan", "Sugar-Free"],
        "nouns":      ["Vitamin D3 Supplement", "Omega-3 Capsules", "Probiotic Tablets", "Magnesium Glycinate", "Collagen Powder", "Zinc Lozenges", "Multivitamin Gummies", "Ashwagandha Extract"],
        "use_cases":  ["daily wellness routines", "immune support", "active individuals", "people over 50"],
        "benefits":   ["Third-party tested for purity and potency", "Free from artificial colours, flavours, and fillers", "Easy to swallow and gentle on the stomach", "Suitable for vegetarians and vegans"]
    },
    "Outdoor": {
        "adjectives": ["All-Weather", "Ultralight", "Heavy-Duty", "Packable", "Waterproof"],
        "nouns":      ["Hiking Boots", "Camping Tent", "Sleeping Bag", "Trekking Poles", "Headtorch", "Dry Bag", "Camping Stove", "Hammock"],
        "use_cases":  ["hiking and trekking", "wild camping", "festival season", "backpacking expeditions"],
        "benefits":   ["Tested in extreme weather conditions", "Packs down small enough to fit in a day bag", "Provides excellent grip and ankle support on uneven terrain", "Made from recycled and sustainably sourced materials"]
    }
}

def make_product_name(category):
    vocab = PRODUCT_VOCAB[category]
    adj   = random.choice(vocab["adjectives"])
    noun  = random.choice(vocab["nouns"])
    return f"{adj} {noun}"

def make_product_description(category, name):
    vocab    = PRODUCT_VOCAB[category]
    use_case = random.choice(vocab["use_cases"])
    benefit  = random.choice(vocab["benefits"])
    return f"The {name} is designed for {use_case}. {benefit}."

print(f"{len(CATEGORIES)} categories, {len(TAGS)} tags defined.")

15 categories, 30 tags defined.


## 8. Generate data

In [10]:
NUM_CUSTOMERS = 2000
NUM_PRODUCTS  = 500
NUM_ORDERS    = 20000

# Customers
customers = [
    {
        "id":      str(uuid.uuid4()),
        "name":    fake.name(),
        "email":   fake.unique.email(),
        "city":    fake.city(),
        "country": fake.country()
    }
    for _ in range(NUM_CUSTOMERS)
]

# Products — realistic names and descriptions per category
products = []
for _ in range(NUM_PRODUCTS):
    category = random.choice(CATEGORIES)
    name     = make_product_name(category)
    products.append({
        "id":          str(uuid.uuid4()),
        "name":        name,
        "description": make_product_description(category, name),
        "price":       round(random.uniform(5.0, 500.0), 2),
        "category":    category
    })

# Tags per product: 2-4 drawn from the vocabulary
product_tags = {
    p["id"]: random.sample(TAGS, k=random.randint(3, 6))
    for p in products
}

# Orders: each order contains a basket of 2-4 products shared by the same order_id.
# This enables co-purchase queries (frequently bought together).
start_date = datetime(2023, 1, 1)
orders = []
for _ in range(NUM_ORDERS):
    order_id   = str(uuid.uuid4())
    customer   = random.choice(customers)
    order_date = (start_date + timedelta(days=random.randint(0, 730))).strftime("%Y-%m-%d")
    basket     = random.sample(products, k=random.randint(2, 4))
    for product in basket:
        orders.append({
            "order_id":    order_id,
            "customer_id": customer["id"],
            "product_id":  product["id"],
            "quantity":    random.randint(1, 5),
            "order_date":  order_date
        })

print(f"Generated {len(customers)} customers, {len(products)} products, {len(orders)} order lines across {NUM_ORDERS} orders.")
print("\nSample products:")
print(tabulate(
    [[p["name"], p["category"], f"${p['price']:.2f}", p["description"]] for p in products[:5]],
    headers    = ["Name", "Category", "Price", "Description"],
    maxcolwidths = [30, 15, 10, 50]
))

Generated 2000 customers, 500 products, 59755 order lines across 20000 orders.

Sample products:
Name                           Category       Price    Description
-----------------------------  -------------  -------  --------------------------------------------------
Ceramic Herb Garden Kit        Home & Garden  $253.32  The Ceramic Herb Garden Kit is designed for
                                                       everyday cooking. Easy to clean and dishwasher
                                                       safe.
Waterproof Compression Sacks   Travel         $335.91  The Waterproof Compression Sacks is designed for
                                                       business travel. TSA-approved and airline carry-on
                                                       compliant.
Ergonomic Monitor Stand        Office         $317.21  The Ergonomic Monitor Stand is designed for remote
                                                       workers. Made from durable and 

## 9. Load data into Aura

In [11]:
def load_customers(tx, batch):
    tx.run("""
        UNWIND $batch AS row
        MERGE (c:Customer {id: row.id})
        SET c.name    = row.name,
            c.email   = row.email,
            c.city    = row.city,
            c.country = row.country
    """, batch=batch)

def load_products(tx, batch):
    tx.run("""
        UNWIND $batch AS row
        MERGE (p:Product {id: row.id})
        SET p.name        = row.name,
            p.description = row.description,
            p.price       = row.price
        MERGE (cat:Category {name: row.category})
        MERGE (p)-[:BELONGS_TO]->(cat)
    """, batch=batch)

def load_tags(tx, product_id, tags):
    tx.run("""
        MATCH (p:Product {id: $product_id})
        UNWIND $tags AS tag_name
        MERGE (t:Tag {name: tag_name})
        MERGE (p)-[:TAGGED_WITH]->(t)
    """, product_id=product_id, tags=tags)

def load_orders(tx, batch):
    tx.run("""
        UNWIND $batch AS row
        MATCH (c:Customer {id: row.customer_id})
        MATCH (p:Product  {id: row.product_id})
        MERGE (c)-[r:PURCHASED {order_id: row.order_id}]->(p)
        SET r.quantity   = row.quantity,
            r.order_date = row.order_date
    """, batch=batch)

BATCH_SIZE = 100

with driver.session() as session:

    customer_batches = range(0, len(customers), BATCH_SIZE)
    for i in tqdm(customer_batches, desc="Loading customers", unit="batch", file=sys.stdout, colour="#1f77b4"):
        session.execute_write(load_customers, customers[i:i+BATCH_SIZE])

    product_batches = range(0, len(products), BATCH_SIZE)
    for i in tqdm(product_batches, desc="Loading products ", unit="batch", file=sys.stdout, colour="#1f77b4"):
        session.execute_write(load_products, products[i:i+BATCH_SIZE])

    for product_id, tags in tqdm(product_tags.items(), desc="Loading tags     ", unit="product", file=sys.stdout, colour="#1f77b4"):
        session.execute_write(load_tags, product_id, tags)

    order_batches = range(0, len(orders), BATCH_SIZE)
    for i in tqdm(order_batches, desc="Loading orders   ", unit="batch", file=sys.stdout, colour="#1f77b4"):
        session.execute_write(load_orders, orders[i:i+BATCH_SIZE])

Loading orders   : 100%|████████████████████████████████████████████████| 598/598 [01:12<00:00,  8.28batch/s]


## 10. Verify load

In [12]:
verify_query = """
MATCH (c:Customer)   WITH count(c) AS customers
MATCH (p:Product)    WITH customers, count(p) AS products
MATCH (cat:Category) WITH customers, products, count(cat) AS categories
MATCH (t:Tag)        WITH customers, products, categories, count(t) AS tags
MATCH ()-[r:PURCHASED]->() RETURN customers, products, categories, tags, count(r) AS order_lines
"""

with driver.session() as session:
    result = session.run(verify_query).single()

rows = [[
    result["customers"],
    result["products"],
    result["categories"],
    result["tags"],
    result["order_lines"]
]]

print(tabulate(rows, headers = ["Customers", "Products", "Categories", "Tags", "Order Lines"]))

  Customers    Products    Categories    Tags    Order Lines
-----------  ----------  ------------  ------  -------------
       2000         500            15      30          59755


## 11. Teardown

In [13]:
driver.close()
print("Connection closed.")

Connection closed.
